# Phase 5 — the work table stays on the terminal ballot

A run starts from a real table, then builds derived tables. The ruler scores both, but their scores live in different journal fields: `work_probe` for the data as given and `tables` for tool-produced artifacts. This notebook walks the terminal choice by hand before any automation enters.

## 1. The old ballot silently dropped a candidate

For log loss, lower is better. Imagine that the work table scored 0.10 and the only derived table scored 0.20. The old terminal path passed only `tables` to `best_table`, so it had no way to return the genuinely better work table.

In [ ]:
from tabular_agent.harness.stop import best_table

work_probe = {"probe_score": 0.10}
derived = [
    {"id": "t1", "path": "tables/t1.parquet", "probe_score": 0.20}
]

old_winner = best_table(derived, "logloss")
assert old_winner["id"] == "t1"
old_winner

## 2. Put the data as given on the same ballot

The fix does not invent a second selector. It gives the existing selector the missing candidate, using the stored work score and the real work-table path. Raw goes first because an exact tie should prefer its empty lineage.

In [ ]:
work = {
    **work_probe,
    "id": "work",
    "path": "data/work.parquet",
}
ballot = [work, *derived]
winner = best_table(ballot, "logloss")
assert winner["id"] == "work"
winner

## 3. The boundary cases express the rule

A tie goes to the work table because it has no fitted feature lineage to replay. A genuinely better derived table still wins. Nothing about model enumeration or blending changes after this table choice.

In [ ]:
tie = [{**derived[0], "probe_score": work_probe["probe_score"]}]
assert best_table([work, *tie], "logloss")["id"] == "work"

better = [{**derived[0], "probe_score": 0.08}]
assert best_table([work, *better], "logloss")["id"] == "t1"
"both boundary cases passed"

## 4. What deliberately remains open

This fix changes only terminal selection. The stop counter still treats the first derived table as the first scored attempt even if it is worse than `work_probe`, and the agent-facing payload still lacks the exact gain against the global incumbent. Those are signal questions, not part of this ballot bug. Keeping them separate lets the regression test say exactly what changed.

The production path is exercised by `test_finalize_ships_the_work_table_when_every_derived_table_is_worse`, which runs the real terminal fits and checks their stored `source_table`.